<a href="https://colab.research.google.com/github/KaanCesur354/CENG467_TakeHome/blob/main/CENG467_TakeHomeQ4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install datasets transformers torch sacrebleu evaluate nltk -q

import random
import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

print("Setup done!")
print("GPU available:", torch.cuda.is_available())

In [ ]:
from datasets import load_dataset
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

dataset = load_dataset("bentrevett/multi30k")

print(dataset)
print("\n--- Sample record ---")
print("English:", dataset["train"][0]["en"])
print("German: ", dataset["train"][0]["de"])
print(f"\nTrain: {len(dataset['train'])}, Val: {len(dataset['validation'])}, Test: {len(dataset['test'])}")

In [ ]:
from collections import Counter

def build_vocab(sentences, max_vocab=10000):
    counter = Counter()
    for sent in sentences:
        counter.update(sent.lower().split())
    vocab = {"<PAD>": 0, "<UNK>": 1, "<SOS>": 2, "<EOS>": 3}
    for word, _ in counter.most_common(max_vocab - 4):
        vocab[word] = len(vocab)
    return vocab

train_en = [ex["en"] for ex in dataset["train"]]
train_de = [ex["de"] for ex in dataset["train"]]
val_en   = [ex["en"] for ex in dataset["validation"]]
val_de   = [ex["de"] for ex in dataset["validation"]]
test_en  = [ex["en"] for ex in dataset["test"]]
test_de  = [ex["de"] for ex in dataset["test"]]

src_vocab = build_vocab(train_en)
tgt_vocab = build_vocab(train_de)
tgt_idx2word = {v: k for k, v in tgt_vocab.items()}

def encode(sentence, vocab, max_len=50):
    tokens = sentence.lower().split()[:max_len]
    ids = [vocab.get(t, 1) for t in tokens]
    ids = [2] + ids + [3]
    ids += [0] * (max_len + 2 - len(ids))
    return ids

print(f"Source (EN) vocab size: {len(src_vocab)}")
print(f"Target (DE) vocab size: {len(tgt_vocab)}")

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

class TranslationDataset(Dataset):
    def __init__(self, src_sentences, tgt_sentences, src_vocab, tgt_vocab, max_len=50):
        self.src = [encode(s, src_vocab, max_len) for s in src_sentences]
        self.tgt = [encode(s, tgt_vocab, max_len) for s in tgt_sentences]

    def __len__(self):
        return len(self.src)

    def __getitem__(self, idx):
        return (
            torch.tensor(self.src[idx], dtype=torch.long),
            torch.tensor(self.tgt[idx], dtype=torch.long)
        )

train_dataset = TranslationDataset(train_en, train_de, src_vocab, tgt_vocab)
val_dataset   = TranslationDataset(val_en,   val_de,   src_vocab, tgt_vocab)
test_dataset  = TranslationDataset(test_en,  test_de,  src_vocab, tgt_vocab)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=64)
test_loader  = DataLoader(test_dataset,  batch_size=64)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Datasets ready!")

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class Encoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.rnn = nn.GRU(embed_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.fc = nn.Linear(hidden_dim * 2, hidden_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        embedded = self.dropout(self.embedding(x))
        outputs, hidden = self.rnn(embedded)
        hidden = torch.tanh(self.fc(torch.cat((hidden[-2], hidden[-1]), dim=1)))
        return outputs, hidden

class Attention(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()
        self.attn = nn.Linear(hidden_dim * 3, hidden_dim)
        self.v = nn.Linear(hidden_dim, 1, bias=False)

    def forward(self, hidden, encoder_outputs):
        src_len = encoder_outputs.shape[1]
        hidden = hidden.unsqueeze(1).repeat(1, src_len, 1)
        energy = torch.tanh(self.attn(torch.cat((hidden, encoder_outputs), dim=2)))
        return F.softmax(self.v(energy).squeeze(2), dim=1)

class Decoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.attention = Attention(hidden_dim)
        self.rnn = nn.GRU(embed_dim + hidden_dim * 2, hidden_dim, batch_first=True)
        self.fc_out = nn.Linear(hidden_dim * 3 + embed_dim, vocab_size)
        self.dropout = nn.Dropout(dropout)

    def forward(self, tgt, hidden, encoder_outputs):
        tgt = tgt.unsqueeze(1)
        embedded = self.dropout(self.embedding(tgt))
        attn_weights = self.attention(hidden, encoder_outputs).unsqueeze(1)
        context = torch.bmm(attn_weights, encoder_outputs)
        rnn_input = torch.cat((embedded, context), dim=2)
        output, hidden = self.rnn(rnn_input, hidden.unsqueeze(0))
        prediction = self.fc_out(torch.cat((output, context, embedded), dim=2).squeeze(1))
        return prediction, hidden.squeeze(0)

class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device

    def forward(self, src, tgt, teacher_forcing_ratio=0.5):
        batch_size, tgt_len = tgt.shape
        tgt_vocab_size = self.decoder.fc_out.out_features
        outputs = torch.zeros(batch_size, tgt_len, tgt_vocab_size).to(self.device)
        encoder_outputs, hidden = self.encoder(src)
        dec_input = tgt[:, 0]
        for t in range(1, tgt_len):
            output, hidden = self.decoder(dec_input, hidden, encoder_outputs)
            outputs[:, t] = output
            teacher_force = torch.rand(1).item() < teacher_forcing_ratio
            dec_input = tgt[:, t] if teacher_force else output.argmax(1)
        return outputs

encoder = Encoder(len(src_vocab), 256, 512)
decoder = Decoder(len(tgt_vocab), 256, 512)
seq2seq = Seq2Seq(encoder, decoder, device).to(device)
print(f"Model parameters: {sum(p.numel() for p in seq2seq.parameters()):,}")

In [ ]:
optimizer = torch.optim.Adam(seq2seq.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss(ignore_index=0)

for epoch in range(10):
    seq2seq.train()
    for src, tgt in train_loader:
        src, tgt = src.to(device), tgt.to(device)
        optimizer.zero_grad()
        output = seq2seq(src, tgt)
        output = output[:, 1:].reshape(-1, output.shape[-1])
        tgt = tgt[:, 1:].reshape(-1)
        loss = criterion(output, tgt)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(seq2seq.parameters(), 1.0)
        optimizer.step()
    print(f"Epoch {epoch+1}/10 done.")

In [ ]:
from transformers import MarianMTModel, MarianTokenizer
model_name = "Helsinki-NLP/opus-mt-en-de"
hel_tokenizer = MarianTokenizer.from_pretrained(model_name)
hel_model = MarianMTModel.from_pretrained(model_name).to(device)
hel_model.eval()

def helsinki_translate(sentences):
    inputs = hel_tokenizer(sentences, return_tensors="pt", padding=True, truncation=True).to(device)
    with torch.no_grad():
        outputs = hel_model.generate(**inputs)
    return hel_tokenizer.batch_decode(outputs, skip_special_tokens=True)

print("Helsinki model ready.")

In [ ]:
def seq2seq_translate(sentences, model, src_vocab, tgt_idx2word, max_len=50):
    model.eval()
    translations = []
    with torch.no_grad():
        for sent in sentences:
            src = torch.tensor([encode(sent, src_vocab, max_len)], dtype=torch.long).to(device)
            enc_out, hidden = model.encoder(src)
            dec_in = torch.tensor([2], dtype=torch.long).to(device)
            res = []
            for _ in range(max_len):
                out, hidden = model.decoder(dec_in, hidden, enc_out)
                top1 = out.argmax(1)
                word = tgt_idx2word.get(top1.item(), "<UNK>")
                if word == "<EOS>": break
                if word not in ("<PAD>", "<SOS>", "<UNK>"): res.append(word)
                dec_in = top1
            translations.append(" ".join(res))
    return translations

seq2seq_preds = seq2seq_translate(test_en[:100], seq2seq, src_vocab, tgt_idx2word)
helsinki_preds = helsinki_translate(test_en[:100])
print("Translations generated.")

In [ ]:
import evaluate
sacrebleu = evaluate.load("sacrebleu")
references = [[r] for r in test_de[:100]]
bleu_s = sacrebleu.compute(predictions=seq2seq_preds, references=references)
bleu_h = sacrebleu.compute(predictions=helsinki_preds, references=references)
print(f"Seq2Seq BLEU: {bleu_s['score']:.2f}")
print(f"Helsinki BLEU: {bleu_h['score']:.2f}")